# Project FORESIGHT — Data Cleaning Pipeline

Memory-efficient cleaning for the large `sales_transaction.csv`.

Expected raw files:
- `data/raw/sales_transaction.csv`
- `data/raw/sku_master.csv`
- `data/raw/inventory_snapshot.csv`
- `data/raw/promotions.csv`

The pipeline creates a SKU-day sales dataset, cleans the supporting tables, and saves an analysis-ready dataset.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

from pathlib import Path

# If notebook is running from the src folder
BASE_DIR = Path.cwd().parent

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SALES_FILE = RAW_DIR / "sales_transaction.csv"
SKU_FILE = RAW_DIR / "sku_master.csv"
INVENTORY_FILE = RAW_DIR / "inventory_snapshot.csv"
PROMOTIONS_FILE = RAW_DIR / "promotions.csv"

CHUNK_SIZE = 250_000

print("Project root:", BASE_DIR)
print("Raw folder:", RAW_DIR)
print("Sales file:", SALES_FILE)
print("Sales file exists:", SALES_FILE.exists())

SALES_FILE = RAW_DIR / "sales_transaction.csv"
SKU_FILE = RAW_DIR / "sku_master.csv"
INVENTORY_FILE = RAW_DIR / "inventory_snapshot.csv"
PROMOTIONS_FILE = RAW_DIR / "promotions.csv"

# Lower this if your RAM usage is high.
CHUNK_SIZE = 250_000

print("Raw:", RAW_DIR)
print("Processed:", PROCESSED_DIR)


Project root: c:\Users\Rasik\Documents\vs code\foresight
Raw folder: c:\Users\Rasik\Documents\vs code\foresight\data\raw
Sales file: c:\Users\Rasik\Documents\vs code\foresight\data\raw\sales_transaction.csv
Sales file exists: True
Raw: c:\Users\Rasik\Documents\vs code\foresight\data\raw
Processed: c:\Users\Rasik\Documents\vs code\foresight\data\processed


In [13]:
def clean_column_names(df):
    df.columns = (
        df.columns.str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
    )
    return df

def show_basic_info(df, name):
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("\nMissing values:")
    print(df.isna().sum())


## 1. Clean `sales_transaction`

The 765 MB transaction file is read in chunks. Transactions are aggregated to the required **SKU × date** grain.

The target for forecasting will later be `units_sold`.


In [14]:
def clean_sales_transactions():
    if not SALES_FILE.exists():
        raise FileNotFoundError(f"Sales file not found: {SALES_FILE}")

    chunks = []
    total_rows = invalid_dates = invalid_quantity = 0

    for n, chunk in enumerate(
        pd.read_csv(SALES_FILE, chunksize=CHUNK_SIZE, low_memory=False),
        start=1
    ):
        print(f"Processing chunk {n}...")
        total_rows += len(chunk)
        chunk = clean_column_names(chunk)

        chunk["date"] = pd.to_datetime(chunk["date"], errors="coerce")
        invalid_dates += int(chunk["date"].isna().sum())

        for col in ["quantity", "unit_price", "total_value", "discount_pct"]:
            if col in chunk.columns:
                chunk[col] = pd.to_numeric(chunk[col], errors="coerce")

        chunk = chunk.dropna(subset=["date", "sku_id", "quantity"])

        invalid_quantity += int((chunk["quantity"] <= 0).sum())
        chunk = chunk[chunk["quantity"] > 0]

        chunk["sku_id"] = chunk["sku_id"].astype(str).str.strip()

        if "channel" in chunk.columns:
            chunk["channel"] = (
                chunk["channel"].astype("string").str.strip().str.lower()
            )

        if "promo_id" in chunk.columns:
            chunk["promo_id"] = (
                chunk["promo_id"].astype("string").str.strip()
            )

        if "total_value" not in chunk.columns:
            chunk["total_value"] = chunk["quantity"] * chunk["unit_price"]

        daily = (
            chunk.groupby(["date", "sku_id"], as_index=False)
            .agg(
                units_sold=("quantity", "sum"),
                revenue=("total_value", "sum"),
                avg_unit_price=("unit_price", "mean"),
                avg_discount_pct=("discount_pct", "mean"),
                transaction_count=("receipt_id", "nunique")
            )
        )

        chunks.append(daily)

    daily_sales = pd.concat(chunks, ignore_index=True)

    # Re-aggregate because the same date/SKU can occur in different chunks.
    daily_sales = (
        daily_sales.groupby(["date", "sku_id"], as_index=False)
        .agg(
            units_sold=("units_sold", "sum"),
            revenue=("revenue", "sum"),
            avg_unit_price=("avg_unit_price", "mean"),
            avg_discount_pct=("avg_discount_pct", "mean"),
            transaction_count=("transaction_count", "sum")
        )
        .sort_values(["sku_id", "date"])
    )

    output = PROCESSED_DIR / "sales_daily_clean.csv"
    daily_sales.to_csv(output, index=False)

    print("\nCompleted.")
    print(f"Raw rows processed: {total_rows:,}")
    print(f"Invalid dates: {invalid_dates:,}")
    print(f"Non-positive quantities: {invalid_quantity:,}")
    print(f"SKU-day rows: {len(daily_sales):,}")
    print("Saved:", output)

    return daily_sales

daily_sales = clean_sales_transactions()
show_basic_info(daily_sales, "Cleaned Daily Sales")


Processing chunk 1...
Processing chunk 2...
Processing chunk 3...
Processing chunk 4...
Processing chunk 5...
Processing chunk 6...
Processing chunk 7...
Processing chunk 8...
Processing chunk 9...
Processing chunk 10...
Processing chunk 11...
Processing chunk 12...
Processing chunk 13...
Processing chunk 14...
Processing chunk 15...
Processing chunk 16...
Processing chunk 17...
Processing chunk 18...
Processing chunk 19...
Processing chunk 20...
Processing chunk 21...
Processing chunk 22...
Processing chunk 23...
Processing chunk 24...
Processing chunk 25...
Processing chunk 26...
Processing chunk 27...
Processing chunk 28...
Processing chunk 29...
Processing chunk 30...
Processing chunk 31...
Processing chunk 32...
Processing chunk 33...
Processing chunk 34...
Processing chunk 35...
Processing chunk 36...
Processing chunk 37...
Processing chunk 38...
Processing chunk 39...
Processing chunk 40...

Completed.
Raw rows processed: 9,972,038
Invalid dates: 0
Non-positive quantities: 0
SKU

## 2. Clean `sku_master`

In [15]:
def clean_sku_master():
    if not SKU_FILE.exists():
        raise FileNotFoundError(f"SKU master not found: {SKU_FILE}")

    df = clean_column_names(pd.read_csv(SKU_FILE, low_memory=False))

    df["sku_id"] = df["sku_id"].astype(str).str.strip()

    for col in ["category", "subcategory"]:
        if col in df.columns:
            df[col] = df[col].astype("string").str.strip()

    if "launch_date" in df.columns:
        df["launch_date"] = pd.to_datetime(df["launch_date"], errors="coerce")

    for col in ["unit_cost", "list_price"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    print("Duplicate SKU IDs:", int(df["sku_id"].duplicated().sum()))

    df = df.drop_duplicates(subset=["sku_id"], keep="first")

    output = PROCESSED_DIR / "sku_master_clean.csv"
    df.to_csv(output, index=False)

    print("Saved:", output)
    return df

sku_master = clean_sku_master()
show_basic_info(sku_master, "Cleaned SKU Master")


Duplicate SKU IDs: 0
Saved: c:\Users\Rasik\Documents\vs code\foresight\data\processed\sku_master_clean.csv

Cleaned SKU Master
Shape: (5000, 7)
Columns: ['sku_id', 'sku_name', 'category', 'subcategory', 'unit_price', 'cost_price', 'brand']

Missing values:
sku_id         0
sku_name       0
category       0
subcategory    0
unit_price     0
cost_price     0
brand          0
dtype: int64


## 3. Clean `inventory_snapshot`

In [18]:
def clean_inventory():
    if not INVENTORY_FILE.exists():
        raise FileNotFoundError(f"Inventory file not found: {INVENTORY_FILE}")

    df = clean_column_names(pd.read_csv(INVENTORY_FILE, low_memory=False))

    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")

    df["sku_id"] = df["sku_id"].astype(str).str.strip()

    for col in [
        "on_hand_units", "on_order_units", "lead_time_days",
        "reorder_point", "safety_stock"
    ]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["date", "sku_id"]).drop_duplicates()
    df = df.sort_values(["sku_id", "date"])

    output = PROCESSED_DIR / "inventory_clean.csv"
    df.to_csv(output, index=False)

    print("Saved:", output)
    return df

inventory = clean_inventory()
show_basic_info(inventory, "Cleaned Inventory")


FileNotFoundError: Inventory file not found: c:\Users\Rasik\Documents\vs code\foresight\data\raw\sales_transactions.csv

## 4. Clean `promotions`

In [ ]:
def clean_promotions():
    if not PROMOTIONS_FILE.exists():
        raise FileNotFoundError(f"Promotions file not found: {PROMOTIONS_FILE}")

    df = clean_column_names(pd.read_csv(PROMOTIONS_FILE, low_memory=False))

    for col in ["promo_id", "sku_id"]:
        if col in df.columns:
            df[col] = df[col].astype("string").str.strip()

    for col in df.columns:
        name = col.lower()
        if "date" in name or "start" in name or "end" in name:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    for col in ["discount_pct", "discount"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.drop_duplicates()

    output = PROCESSED_DIR / "promotions_clean.csv"
    df.to_csv(output, index=False)

    print("Saved:", output)
    return df

promotions = clean_promotions()
show_basic_info(promotions, "Cleaned Promotions")


## 5. Create analysis-ready daily data

In [ ]:
analysis_daily = daily_sales.merge(
    sku_master,
    on="sku_id",
    how="left",
    validate="many_to_one"
)

analysis_daily["year"] = analysis_daily["date"].dt.year
analysis_daily["month"] = analysis_daily["date"].dt.month
analysis_daily["week"] = analysis_daily["date"].dt.isocalendar().week.astype(int)
analysis_daily["quarter"] = analysis_daily["date"].dt.quarter
analysis_daily["day_of_week"] = analysis_daily["date"].dt.dayofweek
analysis_daily["is_weekend"] = (analysis_daily["day_of_week"] >= 5).astype(int)

analysis_daily = analysis_daily.sort_values(["sku_id", "date"])

output = PROCESSED_DIR / "analysis_ready_daily.csv"
analysis_daily.to_csv(output, index=False)

print("Analysis rows:", len(analysis_daily))
print("Saved:", output)
show_basic_info(analysis_daily, "Analysis-Ready Daily Dataset")


## 6. Data-quality report

In [ ]:
datasets = {
    "sales_daily_clean": daily_sales,
    "sku_master_clean": sku_master,
    "inventory_clean": inventory,
    "promotions_clean": promotions,
    "analysis_ready_daily": analysis_daily
}

rows = []

for name, df in datasets.items():
    for col in df.columns:
        rows.append({
            "dataset": name,
            "column": col,
            "rows": len(df),
            "missing_values": int(df[col].isna().sum()),
            "missing_pct": round(df[col].isna().mean() * 100, 2),
            "unique_values": int(df[col].nunique(dropna=True))
        })

quality_report = pd.DataFrame(rows)
quality_file = PROCESSED_DIR / "data_quality_report.csv"
quality_report.to_csv(quality_file, index=False)

display(quality_report)
print("Saved:", quality_file)


## Next step

Do **not** train the forecasting models yet.

After this notebook runs successfully, the next notebook should:
1. Validate the cleaned data.
2. Perform EDA.
3. Aggregate SKU-day demand to SKU-week demand.
4. Analyze seasonality and demand patterns.
5. Build the required seasonal-naive baseline.

The FORESIGHT brief requires a reproducible cleaning pipeline, then EDA/baseline before modeling.
